In [1]:
# ==============================================================================
# PIPELINE DE EXTRAÇÃO E SALVAMENTO DA BASE (NHIS 2025)
# ==============================================================================
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Leitura otimizada direto do CSV
df_bruto = pd.read_csv(Path("../../0-dados_bruto") / "adult25.csv", low_memory=False)
print(f"Base carregada: {df_bruto.shape[0]:,} linhas e {df_bruto.shape[1]} colunas.")



Base carregada: 24,215 linhas e 577 colunas.


In [2]:
df_bruto.head()

,RATCAT_A,INCTCFLG_A,IMPINCFLG_A,HISPALLP_A,RACEALLP_A,ANYDIFF_A,DISAB3_A,SCHDYMSSTC_A,AFNOW,PHQCAT_A,...,LSATIS4_A,PHSTAT_A,HHSTAT_A,RECTYPE,IMPNUM_A,ADHDAGETC_A,WPHINDEX_A,WTFA_A,HHX,POVRATTC_A
0,6,0,1,2,1,1,1,NaN,NaN,3,...,2,3,1,10,1,NaN,26,10636.862,H012128,1.58
1,12,0,1,3,2,1,2,NaN,NaN,1,...,1,4,1,10,1,NaN,34,2996.860,H000617,4.32
2,1,0,0,2,1,1,2,0.0,2.0,2,...,2,3,1,10,1,5.0,26,29032.445,H033243,0.18
3,14,0,0,3,2,2,2,NaN,NaN,1,...,1,4,1,10,1,NaN,30,7875.107,H049140,6.32
4,13,0,1,3,2,2,2,NaN,2.0,1,...,2,2,1,10,1,NaN,33,4824.620,H012797,4.90


Essas são as variáveis de características demográficas, gênero e suporte familiar


A variável AGE65 foi removida por apresentar respostas registradas em apenas 107 dos 24.215 registros. Ela se aplica somente a casos em que a idade numérica não foi informada, por isso sua baixa cobertura não é suficiente para o treinamento do modelo.

In [3]:
ID_DOMICILIO = "ID_DOMICILIO"
PESO_AMOSTRAL = "PESO_AMOSTRAL"
SEXO = "SEXO"
CLASSIFICACAO_URBANO_RURAL = "CLASSIFICACAO_URBANO_RURAL"
REGIAO_GEOGRAFICA = "REGIAO_GEOGRAFICA"
FAIXA_65_MAIS = "FAIXA_65_MAIS"

ESTADO_CIVIL_DECLARADO= "ESTADO_CIVIL_DECLARADO"
ESTADO_CIVIL= "ESTADO_CIVIL"
STATUS_PARENTAL= "STATUS_PARENTAL"
SEXO_CONJUGE= "SEXO_CONJUGE"
SEPARACAO_LEGAL_CONJUGAL= "SEPARACAO_LEGAL_CONJUGAL"
COJUGE_TRABALHA= "COJUGE_TRABALHA"

FLG_IDOSO_FAMILIA= "FLG_IDOSO_FAMILIA"
QTD_ADULTOS_FAMILIA= "QTD_ADULTOS_FAMILIA"
QTD_CRIANÇAS_FAMILIA= "QTD_CRIANÇAS_FAMILIA"
ESCOLARIDADE_MAXIMA_FAMILIA= "ESCOLARIDADE_MAXIMA_FAMILIA"
TEMPO_MORADIA_ANOS= "TEMPO_MORADIA_ANOS"
TIPO_POSSE_IMOVEL= "TIPO_POSSE_IMOVEL"

SUPORTE_SOCIAL= "SUPORTE_SOCIAL"

MAPA_VARIAVEIS_PAPER = {
  # Controles amostrais
    "HHX": ID_DOMICILIO,
    "WTFA_A": PESO_AMOSTRAL,
    #dados pessoais
    "SEX_A": SEXO,
    "URBRRL23": CLASSIFICACAO_URBANO_RURAL,
    "REGION": REGIAO_GEOGRAFICA,
    #"AGE65": FAIXA_65_MAIS,
    #dados família
    "MARITAL_A": ESTADO_CIVIL_DECLARADO,
    "MARSTAT_A": ESTADO_CIVIL,
    "PARSTAT_A": STATUS_PARENTAL, 
    "SPOUSESEX_A": SEXO_CONJUGE,
    "SPOUSEP_A": SEPARACAO_LEGAL_CONJUGAL,
    "SPOUSWRK_A": COJUGE_TRABALHA,
    "OVER65FLG_A": FLG_IDOSO_FAMILIA,
    "PCNTADLT_A": QTD_ADULTOS_FAMILIA,
    "PCNTKIDS_A": QTD_CRIANÇAS_FAMILIA,
    "MAXEDUCP_A": ESCOLARIDADE_MAXIMA_FAMILIA,
    
    #moradia
    "HOUYRSLIV_A": TEMPO_MORADIA_ANOS,
    #"HOUSECOST_A": "dificuldade_custo_moradia", #não estava no paper e não tem em 2025
    "HOUTENURE_A": TIPO_POSSE_IMOVEL, 
   

    "SUPPORT_A": SUPORTE_SOCIAL, #não tem no artigo mas deveria
}

Como cerca de 80% a 85% dos entrevistados no inquérito nasceram nos EUA, mais de 80% da base fica automaticamente sem resposta nessa variável, gerando essa frequência elevada de missingness que observamos.
Por isso substituimos pela variável NATUSBORN_A que pergunta a naturalidade estadunidense do entrevistado.

In [4]:
QTD_ANOS_ESTADOS_UNIDOS= "QTD_ANOS_ESTADOS_UNIDOS"
NATURAL_ESTADOS_UNIDOS = "NATURAL_ESTADOS_UNIDOS"

MAPA_VARIAVEIS_PAPER.update(
    {
    "NATUSBORN_A": NATURAL_ESTADOS_UNIDOS,
    #"YRSINUS_A": QTD_ANOS_ESTADOS_UNIDOS,
    }
)

### Seleção de variáveis de saúde física e bem-estar

Como o objetivo do estudo é investigar padrões e associações entre condições físicas e ansiedade e/ou depressão, a seleção das variáveis considerou sua relação com essa hipótese. A inclusão de uma variável no artigo foi tratada como referência, mas não como justificativa suficiente por si só.

- **`BMICAT_A` — categoria de índice de massa corporal (IMC):** foi considerada como característica relacionada à saúde física. Ela permite explorar associações entre as categorias de IMC e ansiedade e/ou depressão, mas não representa, por si só, uma comorbidade ou um diagnóstico clínico. O codebook informa que o IMC foi calculado a partir das variáveis de altura e peso e que alguns valores extremos foram limitados por razões de confidencialidade.

- **`PHSTAT_A` — estado geral de saúde autorreferido:** pode ser usado como indicador amplo da percepção da própria saúde, classificada de “Excelente” a “Ruim”. Não identifica condições físicas específicas; portanto, deve ser interpretado como uma medida geral de saúde, e não como diagnóstico ou comorbidade.

- **`WEIGHTLBTC_A` — peso corporal em libras:** registra o peso sem calçados, mas não considera a altura. Como o IMC já combina peso e altura, o peso isolado pode trazer informação parcialmente redundante. Por esse motivo, não foi priorizado junto com `BMICATD_A`, a menos que o peso corporal, por si só, seja relevante para uma análise específica. Valores codificados como “Not available” devem ser tratados conforme as orientações do codebook, e não como pesos válidos.

- **`LSATIS4_A` — satisfação com a vida:** mede satisfação geral com a vida, não uma condição física. Como está relacionada ao bem-estar subjetivo e pode se aproximar conceitualmente dos desfechos de ansiedade e depressão, não foi priorizada como preditora na análise principal. Poderá ser explorada separadamente, se houver interesse em estudar sua relação com esses desfechos.

Essas variáveis permitem explorar associações, mas não demonstram que uma condição física cause ansiedade ou depressão. Como os dados são observacionais, os resultados devem ser interpretados como padrões de associação. Quando disponíveis e pertinentes, variáveis que representem condições físicas ou doenças específicas podem ser mais diretamente relacionadas à hipótese de comorbidades do que medidas gerais, como peso ou saúde autorreferida.

In [5]:
AUTOAVALIACAO_SAUDE_GERAL = "AUTOAVALIACAO_SAUDE_GERAL"
CATEGORIA_IMC = "CATEGORIA_IMC"
PESO_LIBRAS = "PESO_LIBRAS"
SATISFACAO_COM_A_VIDA = "SATISFACAO_COM_A_VIDA"




MAPA_VARIAVEIS_PAPER.update(
   {
        #saude geral
        "PHSTAT_A": AUTOAVALIACAO_SAUDE_GERAL, 
        "BMICAT_A": CATEGORIA_IMC,
        #"WEIGHTLBTC_A": PESO_LIBRAS,
        #"LSATIS4_A": SATISFACAO_COM_A_VIDA,
        


   } 
)


### Seleção das variáveis de Colesterol

Mantemos apenas CHLEV_A porque ela representa o histórico de colesterol elevado — o conceito que queremos analisar — e é mapeada para HISTORICO_COLESTEROL no notebook. CHLMED_A, por sua vez, registra o uso de medicamentos para colesterol: é uma variável de tratamento, não uma medida equivalente do histórico da condição. Por isso, ela permanece fora do mapeamento. A decisão também evita confundir o histórico de colesterol com o tratamento recebido.

In [6]:
HISTORICO_COLESTEROL ="HISTORICO_COLESTEROL"
REMEDIO_COLESTEROL= "REMEDIO_COLESTEROL"


MAPA_VARIAVEIS_PAPER.update(
   {

        "CHLEV_A": HISTORICO_COLESTEROL,
        #"CHLMED_A": REMEDIO_COLESTEROL,

   } 
)

### Seleção das variáveis de Diabetes

Substituímos apenas por DIBEV_A porque ela mede o histórico de diabetes — o conceito que queremos representar — e, conforme os dados informados, não tem valores ausentes. No notebook, ela é mapeada para HISTORICO_DIABETES (02_preparacao_e_limpeza_2025_mdd.ipynb).

As outras variáveis não são alternativas equivalentes: DIBPILL_A registra uso de comprimidos para diabetes e tem 76,85% de valores ausentes; DIBINSSTYR_A trata de interrupção de insulina no último ano e tem 99,897% ausentes. Além da baixa cobertura, elas descrevem tratamentos ou comportamentos específicos, não se a pessoa tem histórico de diabetes. Os ausentes podem decorrer da aplicabilidade ou do fluxo das perguntas; não devem ser interpretados automaticamente como “não”.

Assim, usar apenas DIBEV_A preserva a cobertura da amostra e mantém a variável alinhada ao conceito analisado, sem confundir diagnóstico/histórico com tratamento.

In [7]:
HISTORICO_DIABETES = "HISTORICO_DIABETES"
INTERROMPEU_INSULINA_PRM_ANO = "INTERROMPEU_INSULINA_PRM_ANO"
MED_DIABETES_ORAL = "MED_DIABETES_ORAL"


MAPA_VARIAVEIS_PAPER.update(
   {
       
      "DIBEV_A": HISTORICO_DIABETES,
      #"DIBPILL_A": MED_DIABETES_ORAL, # no paper usou dibpill mas tem vários outros tratamentos
      #"DIBINSSTYR_A": INTERROMPEU_INSULINA_PRM_ANO,

   } 
)

### Seleção das variáveis de Hipertensão

Mantemos apenas **HYPEV_A**, que representa o histórico de hipertensão — o conceito que queremos analisar. A variável **HYPMED_A**, por sua vez, registra se a pessoa toma medicamentos para controlar a hipertensão; portanto, descreve um tratamento, e não a presença ou o histórico da condição. Observamos que, diferentemente do que ocorre com outras condições de saúde consideradas no artigo, a seleção não inclui a variável sobre uso de medicamentos (**HYPMED_A**). Para manter o foco no histórico da condição e evitar misturá-lo com o tratamento recebido, **HYPMED_A** permanece fora do mapeamento.

In [8]:
HISTORICO_HIPERTENSAO ="HISTORICO_HIPERTENSAO"
#MED_HIPERTENSAO = "MED_HIPERTENSAO"

MAPA_VARIAVEIS_PAPER.update(
   {

        "HYPEV_A": HISTORICO_HIPERTENSAO,
         #"HYPMED_A": MED_HIPERTENSAO,
   } 
)


### Seleção das variáveis de Artrite

Mantemos **ARTHEV_A** como medida do histórico de artrite, pois representa diretamente a condição de interesse e é mapeada para **HISTORICO_ARTRITE**. Diferentemente de algumas outras condições de saúde, o survey não inclui uma pergunta sobre o uso de medicamentos para artrite. Assim, a seleção considera apenas o histórico da condição, sem uma variável de tratamento correspondente.

In [9]:
HISTORICO_ARTRITE = "HISTORICO_ARTRITE"

MAPA_VARIAVEIS_PAPER.update(
   {

        "ARTHEV_A": HISTORICO_ARTRITE,

   } 
)


### Seleção das variáves de lesões

As variáveis `REPWRKCAUS_A`, `REPSTRAIN_A` e `REPLIMIT_A`, mencionadas no artigo para representar aspectos relacionados a lesões por esforço repetitivo (LER/DORT), não foram selecionadas para a análise de 2025 porque não estão disponíveis no survey NHIS desse ano. Embora constem no questionário de 2020, sua ausência no instrumento de 2025 impede uma comparação direta com base nessas variáveis.

Também foi selecionada a variável `TBIHLSBMC_A`, que aborda outro tipo de lesão: sintomas nos últimos 12 meses após uma pancada ou solavanco na cabeça, como dor de cabeça, sensibilidades, problemas de equilíbrio ou alterações de humor/comportamento. **Essa variável não substitui as medidas de LER/DORT; ela representa uma dimensão distinta de lesões incluída no bloco.**

In [10]:
LER_DORT_RELACIONADA_AO_TRABALHO = "LER_DORT_RELACIONADA_AO_TRABALHO"
LESAO_POR_ESFORCO_REPETITIVO_ULTIMOS_3_MESES = "LESAO_POR_ESFORCO_REPETITIVO_ULTIMOS_3_MESES"
lIMITACAO_ATIVIDADES_POR_LER_DORT_ULTIMOS_3_MESES = "lIMITACAO_ATIVIDADES_POR_LER_DORT_ULTIMOS_3_MESES"

CONCUSSAO_SINTOMAS_POS_TRAUMA ="CONCUSSAO_SINTOMAS_POS_TRAUMA"

MAPA_VARIAVEIS_PAPER.update(
   {
      #"REPWRKCAUS_A": "LER_DORT_RELACIONADA_AO_TRABALHO",  # não disponível em 2025
      #"REPSTRAIN_A": "LESAO_POR_ESFORCO_REPETITIVO_ULTIMOS_3_MESES",  # não disponível em 2025
      #"REPLIMIT_A": "lIMITACAO_ATIVIDADES_POR_LER_DORT_ULTIMOS_3_MESES",  # não disponível em 2025

      "TBIHLSBMC_A": CONCUSSAO_SINTOMAS_POS_TRAUMA

   } 
)


### Otimização do Bloco de Mobilidade e Dificuldades Motoras

**Problema Identificado:**
As variáveis detalhadas de caminhada e uso de escadas (`NOEQWLK13M_A`, `EQWLK13M_A`, `NOEQSTEPS_A` e `EQSTEPS_A`) apresentavam **taxas de *missingness* elevadas (entre 90.4% e 94.1%)** na amostra de 24.215 participantes[cite: 5]:

* **Caminhar 500m sem aparelho (`NOEQWLK13M_A`):** 94.08% de nulos
* **Caminhar 500m com aparelho (`EQWLK13M_A`):** 94.23% de nulos
* **Andar dificuldade degraus sem aparelho (`NOEQSTEPS_A`):** 90.51% de nulos
* **Andar dificuldade degraus com aparelho (`EQSTEPS_A`):** 93.11% de nulos

Essa escassez de dados ocorre devido aos saltos condicionais (*skip patterns*) do questionário do CDC, que dividem a população entre quem usa e não usa equipamentos de auxílio. Aplicar imputação estatística sobre esses campos geraria ruído severo e multicolinearidade nos modelos.

**Solução Aplicada:**
1. **Substituição por Variável Geral (`DIFF_A`):** Eliminamos as 4 colunas detalhadas e adotamos a variável nativa de triagem **`DIFF_A`** (`DIFICULDADE_LOCOMOCAO_GERAL`), aplicada diretamente a toda a população do inquérito.
2. **Integridade da Base:** A variável `DIFF_A` possui **0% de *missingness* estrutural**, garantindo cobertura completa para todos os 24.215 participantes.
3. **Padronização e Escala:** A variável foi recodificada para a escala de $0$ a $3$ ($0 = \text{Sem dificuldade}$ até $3 = \text{Incapaz}$), mantendo o sinal clínico de limitação motora de forma coesa para os algoritmos de Machine Learning.

In [11]:
DIFICULDADE_LOCOMOCAO = "DIFICULDADE_LOCOMOCAO"

ANDAR_DIFICULDADE_500M_SEM_APARELHO = "ANDAR_DIFICULDADE_500M_SEM_APARELHO"
ANDAR_DIFICULDADE_DEGRAUS_SEM_APARELHO = "ANDAR_DIFICULDADE_DEGRAUS_SEM_APARELHO"
ANDAR_DIFICULDADE_500M_COM_APARELHO = "ANDAR_DIFICULDADE_500M_COM_APARELHO"
ANDAR_DIFICULDADE_DEGRAUS_COM_APARELHO = "ANDAR_DIFICULDADE_DEGRAUS_COM_APARELHO"
ANDAR_NECESSITA_AJUDA_OUTRA_PESSOA = "ANDAR_NECESSITA_AJUDA_OUTRA_PESSOA"


MAPA_VARIAVEIS_PAPER.update({
    "DIFF_A": DIFICULDADE_LOCOMOCAO

    #Difivuldade de andar
    #"NOEQWLK13M_A": ANDAR_DIFICULDADE_500M_SEM_APARELHO,
    #"NOEQSTEPS_A": ANDAR_DIFICULDADE_DEGRAUS_SEM_APARELHO,
    #"EQWLK13M_A": ANDAR_DIFICULDADE_500M_COM_APARELHO,
    #"EQSTEPS_A": ANDAR_DIFICULDADE_DEGRAUS_COM_APARELHO,
    #"PERASST_A": ANDAR_NECESSITA_AJUDA_OUTRA_PESSOA,
})



### Seleção da variável de limitação para o trabalho por motivos de saúde

Mantemos **SOCWRKLIM_A** como indicador de limitação para trabalhar devido a um problema de saúde, conforme sua descrição no survey (*“Work limited due to health problem”*). A variável registra o impacto funcional da saúde na vida da pessoa e complementa as informações sobre histórico de condições de saúde. Ela é mapeada para **LIMITACAO_TRABALHO_POR_SAUDE**.

In [12]:
LIMITACAO_TRABALHO_POR_SAUDE ="LIMITACAO_TRABALHO_POR_SAUDE"

MAPA_VARIAVEIS_PAPER.update({
    "SOCWRKLIM_A": LIMITACAO_TRABALHO_POR_SAUDE

})


### Seleção da variável de deficiência

Incluímos **DISAB3_A** como indicador de deficiência, pois resume dificuldades funcionais em seis domínios: visão, audição, mobilidade, comunicação, cognição e autocuidado. Conforme o codebook do NHIS, a classificação considera deficiência quando a pessoa relata “muita dificuldade” ou “não consegue realizar” ao menos uma atividade em um desses domínios. A variável oferece uma medida geral das limitações funcionais e pode contribuir para caracterizar aspectos de saúde e funcionamento dos participantes, sem representar, por si só, um diagnóstico específico. 
Embora **DISAB3_A** não tenha sido incluída no artigo científico, ela foi adicionada a esta análise por representar um indicador geral de deficiência baseado em dificuldades funcionais. Essa informação pode ajudar a caracterizar os participantes e a investigar sua relação com o desfecho estudado. Sua inclusão constitui uma extensão da seleção de variáveis apresentada no artigo.

In [13]:
DEFICIENTE ="DEFICIENTE"

MAPA_VARIAVEIS_PAPER.update({
    "DISAB3_A": DEFICIENTE

})


A variável ASTILL_A diz se a pessoa tem histórico de Asma e ainda tem asma, tendo frequencia de 3.491 registros de 24.215, 14.42%.
Fica a dúvida: utilizamos ASEV_A que diz se a pessoa teve asma ou não ou usamos a ASTILL_A e botamos o valor "não" para todos os registros vazios?
Pessoas que tem asma hoje são 2062 => 8.52%

In [14]:
HISTORICO_ASMA = "HISTORICO_ASMA"
ASMA_ATIVA_ATUALMENTE ="ASMA_ATIVA_ATUALMENTE"

MAPA_VARIAVEIS_PAPER.update(
   {
      "ASTILL_A": ASMA_ATIVA_ATUALMENTE,
      #"ASEV_A": HISTORICO_ASMA
   }
) 

### Seleção das variáveis de Dor Crônica

A variável PAIAPG3M_A que aparece no paper pergunta se a pessoa teve dores abdominais. Apesar de ter relativamente poucos valores nulos: 8.981 de 24215 (37%), 92% dos 15.234 restantes responderam "Not at all" ou "A little" para o nível de dor abdominal sentida.

As respostas:
"Somewhere in between a little and a lot": 529
"A lot" : 606

São apenas 2,1% e 2,5% respectivamente. 


Portanto a variável PAIAPG3M_A será substituida por PAIFRQ3M_A: A frequencia que a pessoa esteve com dor nos ultimos 3 meses.

PAIAPG3M_A é diluida em diversas perguntas que especificam a parte do corpo que a pessoa sente dor cronica. 

PAIBACK3M_A: Coluna
PAIULMB3M_A: Mãos
PAILLMB3M_A: Cintura
PAIHDFC3M_A: Cabeça
PAIAPG3M_A:  Abdominal
PAITOOTH3M_A:Dente ou mandibula

Por isso a pouca quantidade de valores relevantes

In [15]:
DOR_CRONICA = "DOR_CRONICA"


MAPA_VARIAVEIS_PAPER.update(
    {
        "PAIFRQ3M_A": DOR_CRONICA
        
    }
)

### Seleção das variáveis de vacinação

As variáveis de vacinação foram excluídas por não apresentarem relação relevante com o problema investigado e por possuírem muitos dados ausentes. Mantê-las poderia reduzir a quantidade de observações disponíveis para a análise, sem uma contribuição clara para o modelo.

| Variável | Decisão | Justificativa resumida |
|---|---|---|
| **SHTFLU12M_A** | Excluir | Indica se a pessoa recebeu a vacina contra gripe nos últimos 12 meses. Foi considerada pouco relacionada ao problema investigado e apresenta muitos dados ausentes. |
| **SHTFLUY_A** | Excluir | Registra o ano da vacinação contra gripe entre as pessoas que responderam “Sim” à pergunta sobre vacinação nos últimos 12 meses. Seus valores se limitam a 2024 e 2025; portanto, não representa o histórico geral de vacinação. |
| **SHTFLUM_A** | Excluir | Informa o mês da vacinação contra gripe no ano correspondente. É uma informação temporal específica, sem contribuição relevante para a análise proposta. |
| **SHTPNEUNB_A** | Excluir | Refere-se à quantidade de vacinas contra pneumonia e não foi considerada relevante para o problema investigado; também apresenta muitos dados ausentes. |
| **SHTTETANUS_A** | Excluir | Indica vacinação contra tétano nos últimos dez anos. Foi excluída por sua baixa pertinência para a análise e pela quantidade de dados ausentes. |

A exclusão busca manter no conjunto de análise variáveis mais pertinentes ao problema e evitar que informações de vacinação com cobertura limitada reduzam a quantidade de participantes disponíveis. A ausência de resposta nessas variáveis não deve ser interpretada automaticamente como ausência de vacinação.

### Seleção das variáveis Câncer
As variáveis: CERVIAGETC_A, PANCRAGETC_A, ESOPHAGETC_A, LUNGAGETC_A, LIVERAGETC_A, SKNDKAGETC_A, SKNDKAGETC_A, THYROAGETC_A, STOMAAGETC_A,    foram retiradas devido as sua grande quantidade de valores nulos, sendo impossível a sua utilização em treinamento pois não pode ser preenchida com valores Moda.

Dos 24.215 mil registros as variáveis apresentaram-se apenas:

CERVIAGETC_A : 13
PANCRAGETC_A : 13
ESOPHAGETC_A : 17
LUNGAGETC_A  : 120
LIVERAGETC_A : 22
SKNDKAGETC_A : 114
THYROAGETC_A : 108
STOMAAGETC_A : 20

Essas variáveis foram substituidas pela variável CANEV_A : "já teve cancer?"
CANEV_A foi preenchida em todos os registros, com a seguinte frequencia de respostas abaixo

Yes 3378  13.95%
No  20804 85.91%


In [16]:
#IDADE_CANCER_COLO_UTERO = "IDADE_CANCER_COLO_UTERO"
#IDADE_CANCER_PANCREAS ="IDADE_CANCER_PANCREAS"
#IDADE_CANCER_ESOFAGO = "IDADE_CANCER_ESOFAGO"
#IDADE_CANCER_PULMAO = "IDADE_CANCER_PULMAO"
#IDADE_CANCER_FIGADO = "IDADE_CANCER_FIGADO"
#IDADE_CANCER_PELE_INDETERMINADO = "IDADE_CANCER_PELE_INDETERMINADO"
#IDADE_CANCER_TIREOIDE = "IDADE_CANCER_TIREOIDE"
#IDADE_CANCER_ESTOMAGO = "IDADE_CANCER_ESTOMAGO"

TEVE_CANCER = "TEVE_CANCER"


MAPA_VARIAVEIS_PAPER.update(
   {
        #câncer

        #"CERVIAGETC_A": IDADE_CANCER_COLO_UTERO,
        #"PANCRAGETC_A": IDADE_CANCER_PANCREAS, #
        #"ESOPHAGETC_A": IDADE_CANCER_ESOFAGO, #
        #"LUNGAGETC_A": IDADE_CANCER_PULMAO,
        #"LIVERAGETC_A": IDADE_CANCER_FIGADO,
        #"SKNDKAGETC_A": IDADE_CANCER_PELE_INDETERMINADO,
        #"THYROAGETC_A": IDADE_CANCER_TIREOIDE,
        #"STOMAAGETC_A": IDADE_CANCER_ESTOMAGO,
         "CANEV_A": TEVE_CANCER
   } 
)

As variáveis WELLNESS_A e WELLVIS_A foram substituídas por LASTDR_A, que registra há quanto tempo o adulto consultou um médico ou profissional de saúde. LASTDR_A se aplica a todos os adultos e possui respostas para os 24.215 registros, enquanto WELLNESS_A se aplica apenas a quem consultou um médico e WELLVIS_A a um grupo ainda mais restrito, resultando em muitos valores ausentes. Assim, LASTDR_A oferece uma medida mais abrangente e com maior cobertura sobre a utilização de serviços de saúde.

In [17]:
TEMPO_ULTIMO_CHECKUP_GERAL="TEMPO_ULTIMO_CHECKUP_GERAL"
ULTIMA_VISITA_FOI_CHECKUP="ULTIMA_VISITA_FOI_CHECKUP"
ULTIMA_VISITA_MEDICO = "ULTIMA_VISITA_MEDICO"

VISITAS_PRONTO_ATENDIMENTO_12M="VISITAS_PRONTO_ATENDIMENTO_12M"
VISITAS_EMERGENCIA_HOSPITALAR_12M="VISITAS_EMERGENCIA_HOSPITALAR_12M"
VISITAS_EXAME_VISTA_12M="VISITAS_EXAME_VISTA_12M"

MAPA_VARIAVEIS_PAPER.update(
    {
   
        #plano de saude
        "LASTDR_A": ULTIMA_VISITA_MEDICO,
    
        "URGCC12MTC_A": VISITAS_PRONTO_ATENDIMENTO_12M,
        "EMERG12MTC_A": VISITAS_EMERGENCIA_HOSPITALAR_12M,
        "EYEEX12M_A": VISITAS_EXAME_VISTA_12M,
        #"WELLVIS_A": TEMPO_ULTIMO_CHECKUP_GERAL,
        #"WELLNESS_A": ULTIMA_VISITA_FOI_CHECKUP, #meio redundante

        #"INJSAWDOC_A": "atendimento_medico_por_lesao", não tem e, 2025
    }
)


A variável FLUNCH12M1_A foi removida por ter respostas registradas em apenas 4.712 dos 24.215 registros. Ela se aplica somente a adultos que vivem em famílias com crianças de 5 a 17 anos, o que limita sua cobertura na base e sua utilidade para o treinamento do modelo.

In [18]:
MERENDA_ESCOLAR_GRATUITA = "MERENDA_ESCOLAR_GRATUITA"
CATEGORIA_RAZAO_POBREZA= "CATEGORIA_RAZAO_POBREZA"
RAZAO_RENDA_POBREZA= "RAZAO_RENDA_POBREZA"
RENDA_INVESTIMENTOS="RENDA_INVESTIMENTOS"


MAPA_VARIAVEIS_PAPER.update(
    {
        #economicos
        #"FLUNCH12M1_A": MERENDA_ESCOLAR_GRATUITA,
        "RATCAT_A": CATEGORIA_RAZAO_POBREZA,
        "POVRATTC_A": RAZAO_RENDA_POBREZA,
        "INCINTER_A":RENDA_INVESTIMENTOS,
        
        
        
        #"RSNHICOST_A": SEM_PLANO_POR_CUSTO_INACESSIVEL,

        #plano de saude

    }
)

### Seleção das variáveis de cobertura de saúde

**MEDICARE_A** e **PRIVATE_A** foram mantidas como variáveis de contexto, pois a cobertura de saúde pode influenciar o acesso a diagnóstico, tratamento e acompanhamento. **PAYWORRY_A** também foi mantida como indicador de estresse financeiro relacionado a despesas médicas.

As demais variáveis foram excluídas por medirem uso de serviços ou detalhes específicos de planos que não são centrais à análise. Algumas se aplicam apenas a grupos restritos; portanto, seus valores ausentes podem indicar que a pergunta não se aplicava ao participante, e não necessariamente que houve perda de resposta.

| Variável | Decisão | Justificativa resumida |
|---|---|---|
| **VAHOSP_A** | Excluir | Mede atendimento recebido ou pago pela VA nos últimos 12 meses, não uma condição de saúde. |
| **PLN1PAY2_A** | Excluir | Informa se empregador ou sindicato paga o primeiro plano privado. É um detalhe sobre o pagamento do plano; **PRIVATE_A** foi mantida como indicador mais geral de cobertura privada. |
| **PRDEDUC1_A** | Excluir | Refere-se à franquia anual do primeiro plano privado, um detalhe do plano e não uma medida de saúde. |
| **PRDEDUC2_A** | Excluir | Refere-se à franquia de um segundo plano privado; apenas 317 participantes têm resposta registrada (1,31%). |
| **OPDEDUC_A** | Excluir | Refere-se à franquia de um plano estadual; apenas 116 participantes têm resposta registrada (0,48%). |
| **HINOTMYR_A** | Excluir | Registra meses sem cobertura em um grupo específico e, conforme o codebook, não deve ser usada para estimar quem tem cobertura. |
| **PAYNOBLLNW_A** | Excluir | Apenas 2.420 dos 24.215 registros têm respostas codificadas. A pergunta se aplica a um grupo específico, o que limita sua cobertura para o treinamento do modelo. |
| **HISTOPJOB_A** | Excluir | Tem respostas registradas em apenas 829 dos 24.215 registros e se aplica somente a um grupo específico de adultos sem cobertura. |
| **RSNHICOST_A** | Excluir | Apenas 1.487 dos 24.215 registros têm resposta “Sim” ou “Não”. A variável se aplica somente a adultos sem cobertura de saúde. |

A seleção mantém variáveis gerais de cobertura e estresse financeiro relacionado às despesas médicas, enquanto exclui informações muito específicas ou com cobertura limitada na base. Nas perguntas de universo restrito, os valores ausentes não devem ser interpretados automaticamente como respostas negativas.


In [19]:
PLANO_PUBLICO = "PLANO_PUBLICO"
PLANO_PRIVADO = "PLANO_PRIVADO"

PLANO_ESTADUAL_DEDUTIVEL="PLANO_ESTADUAL_DEDUTIVEL"
PLANO_PRIVADO_DEDUTIVEL="PLANO_PRIVADO_DEDUTIVEL"
PLANO_PRIVADO_DEDUTIVEL2="PLANO_PRIVADO_DEDUTIVEL2"
PLANO_MILITAR_SAUDE="PLANO_MILITAR_SAUDE"
PLANO_PATRAO_SINDICATO_PAGAM="PLANO_PATRAO_SINDICATO_PAGAM"
PLANO_MESES_SEM="PLANO_MESES_SEM"

SEM_PLANO_POR_CUSTO_INACESSIVEL="SEM_PLANO_POR_CUSTO_INACESSIVEL"
PERDEU_PLANO_DEMITIDO_OU_TROCOU_EMPREGO="PERDEU_PLANO_DEMITIDO_OU_TROCOU_EMPREGO"

ESTRESSE_FINANCEIRO_MEDICO="ESTRESSE_FINANCEIRO_MEDICO"

MAPA_VARIAVEIS_PAPER.update(
    {
        #plano de saude
        "MEDICARE_A" : PLANO_PUBLICO,
        "PRIVATE_A": PLANO_PRIVADO,
        "PAYWORRY_A": ESTRESSE_FINANCEIRO_MEDICO,
        
        #"OPDEDUC_A": PLANO_ESTADUAL_DEDUTIVEL,

        #"OPDEDUC_A": PLANO_ESTADUAL_DEDUTIVEL,
        #"PRDEDUC1_A": PLANO_PRIVADO_DEDUTIVEL,
        #"PRDEDUC2_A": PLANO_PRIVADO_DEDUTIVEL2,
        #"VAHOSP_A": PLANO_MILITAR_SAUDE,
        #"HISTOPJOB_A":PERDEU_PLANO_DEMITIDO_OU_TROCOU_EMPREGO,
        #"PLN1PAY2_A": PLANO_PATRAO_SINDICATO_PAGAM,
        #"HINOTMYR_A": PLANO_MESES_SEM, #
        #"PAYNOBLLNW_A": DIVIDAS_MEDICAS_EM_ABERTO,
        
        #"PLN1PAY2_A": PLANO_PATRAO_SINDICATO_PAGAM,
        #"PRDEDUC1_A": PLANO_PRIVADO_DEDUTIVEL1,
        #"PRDEDUC2_A": PLANO_PRIVADO_DEDUTIVEL2,
        
    }
)

### Seleção das variáveis relacionadas ao tabagismo

As variáveis **SMKNOW_A**, **CIGNOW_A**, **SMK30D_A**, **CIG30D_A** e **ECIGNOW_A** foram excluídas por terem respostas registradas apenas para subconjuntos dos 24.215 participantes. Seus universos de aplicação são restritos a pessoas com histórico de tabagismo ou uso de cigarros eletrônicos e, em alguns casos, a fumantes atuais com um padrão específico de consumo. Essa cobertura limitada reduz sua representatividade na amostra completa.

| Variável | Registros não faltantes | Percentual dos 24.215 registros | Decisão | Justificativa |
|---|---:|---:|---|---|
| **SMKNOW_A** | 8.369 | 34,56% | Excluir | Indica o status atual de tabagismo e se aplica somente a adultos que já fumaram 100 cigarros ou mais. |
| **CIGNOW_A** | 1.626 | 6,71% | Excluir | Informa a quantidade de cigarros fumados por dia e se aplica somente a quem fuma todos os dias. |
| **SMK30D_A** | 609 | 2,51% | Excluir | Registra em quantos dos últimos 30 dias a pessoa fumou; aplica-se somente a quem fuma alguns dias. |
| **CIG30D_A** | 563 | 2,33% | Excluir | Informa a quantidade de cigarros consumidos nos dias em que a pessoa fumou no último mês; aplica-se a um grupo restrito de fumantes ocasionais. |
| **ECIGNOW_A** | 4.280 | 17,67% | Excluir | Registra o uso atual de cigarros eletrônicos e se aplica somente a quem já usou esses produtos. |

A seleção mantém **SMKEV_A**, renomeada para **JA_FUMANTE**, como indicador geral de histórico de tabagismo. Ela não substitui as informações sobre consumo atual de cigarros ou uso de cigarros eletrônicos registradas pelas variáveis excluídas. Como algumas perguntas se aplicam somente a grupos específicos, os valores ausentes não devem ser interpretados automaticamente como respostas negativas.

In [20]:
#FUMA_CIGARRO_ATUALMENTE= "FUMA_CIGARRO_ATUALMENTE"
#QTD_CIGARROS_DIA= "QTD_CIGARROS_DIA"
#DIAS_FUMADOS_MES= "DIAS_FUMADOS_MES"
#QTD_CIGARRO_MES= "QTD_CIGARRO_MES"
#USA_CIGARRO_ELETRONICO_ATUALMENTE= "USA_CIGARRO_ELETRONICO_ATUALMENTE"
JA_FUMANTE= "JA_FUMANTE"

MAPA_VARIAVEIS_PAPER.update(
    {
        #SMKNOW_A": FUMA_CIGARRO_ATUALMENTE,
        #CIGNOW_A": QTD_CIGARROS_DIA,
        #SMK30D_A": DIAS_FUMADOS_MES,
        #CIG30D_A": QTD_CIGARRO_MES,
        #ECIGNOW_A": USA_CIGARRO_ELETRONICO_ATUALMENTE,
        "SMKEV_A": JA_FUMANTE,
    }
)




### Seleção das variáveis relacionadas à ansiedade

Serão mantidos dois grupos complementares. `ANXEV_A`, `ANXFREQ_A` e `ANXLEVEL_A` registrarão histórico, frequência e intensidade autorrelatados; `ANXMED_A` indicará o uso de medicação para sentimentos de preocupação, nervosismo ou ansiedade. As variáveis `GAD71_A` a `GAD77_A` descreverão a frequência de sete sintomas nos últimos 14 dias, enquanto `GADCAT_A` representará a classificação de severidade derivada da escala GAD.

A inclusão dos itens de sintomas e da categoria de severidade permitirá explorar perfis por clusterização. Como `GADCAT_A` resume os próprios itens GAD, a utilização simultânea da categoria e de todos os itens poderá contar a mesma informação mais de uma vez. Na clusterização, serão comparadas representações com os itens e com a categoria, e a ponderação escolhida será documentada. Histórico, intensidade e uso de medicação representarão aspectos distintos dos sintomas recentes, incluindo diagnóstico, acesso e tratamento, e serão considerados conforme a definição dos perfis.

Na etapa supervisionada, a base completa será preservada como fonte, e será definida uma lista de preditores específica para cada alvo. O próprio alvo e as variáveis que o definirem ou dele forem derivadas serão excluídos dos preditores. Caso o alvo seja um diagnóstico de ansiedade, sintomas GAD, categoria de severidade, histórico e tratamento serão avaliados como possíveis fontes de vazamento, conforme a definição do alvo. Essa seleção será realizada antes do ajuste e dentro do fluxo de validação.

In [21]:



HISTORICO_TRANSTORNO_ANSIEDADE= "HISTORICO_TRANSTORNO_ANSIEDADE"
MED_ANSIEDADE= "MED_ANSIEDADE"
FREQ_ANSIEDADE= "FREQ_ANSIEDADE"
INTENSIDADE_ANSIEDADE= "INTENSIDADE_ANSIEDADE"

FREQ_ANSIEDADE_2SEM= "FREQ_ANSIEDADE_2SEM"
FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM= "FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM"
FREQ_PREOCUPACAO_DESNECESSARIA_2SEM= "FREQ_PREOCUPACAO_DESNECESSARIA_2SEM"
FREQ_DIFICULDADE_DESCANSAR_2SEM= "FREQ_DIFICULDADE_DESCANSAR_2SEM"
FREQ_AGITACAO_INQUIETA= "FREQ_AGITACAO_INQUIETA"
FREQ_IRRITABILIDADE_2SEM = "FREQ_IRRITABILIDADE_2SEM"
FREQ_MEDO_2SEM = "FREQ_MEDO_2SEM"
SEVERIDADE_ANSIEDADE = "SEVERIDADE_ANSIEDADE"

MAPA_VARIAVEIS_PAPER.update(
    {

    "ANXEV_A": HISTORICO_TRANSTORNO_ANSIEDADE,
    "ANXMED_A": MED_ANSIEDADE, #pode enviesar muito!
    "ANXFREQ_A": FREQ_ANSIEDADE,
    "ANXLEVEL_A": INTENSIDADE_ANSIEDADE,

    "GAD71_A": FREQ_ANSIEDADE_2SEM,
    #"NERVOUS_A": "k6_freq_nervosismo",
    "GAD72_A": FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM,
    "GAD73_A": FREQ_PREOCUPACAO_DESNECESSARIA_2SEM,
    "GAD74_A": FREQ_DIFICULDADE_DESCANSAR_2SEM,
    "GAD75_A": FREQ_AGITACAO_INQUIETA, # não tinha no paper mas é de ansiedade
    "GAD76_A": FREQ_IRRITABILIDADE_2SEM,
    "GAD77_A": FREQ_MEDO_2SEM,
    "GADCAT_A": SEVERIDADE_ANSIEDADE,

    #"GADCAT_A": "escala_sintomas_ansiedade", #pode enviesar muito!
    #"K6SPD_A": "sofrimento_psicologico_k6", #não tem no paper nem 2025. Pode enviesar?
    ##############################################################################################################
    
    # Segurança alimentar: INFELIZMENTE NÃO TEM EM 2025
    #"FDSLESS_A": "comeu_menos_que_devia", 
    #"FDSWEIGHT_A": "perdeu_peso_restricao_alimentar", 
    #"FDSSKIP_A": "diminuiu_ou_pulou_refeicoes",
    #"FDSHUNGRY_A": "passou_fome_por_falta_dinheiro",
    
    }
)




### Seleção das variáveis relacionadas à depressão

`DEPEV_A`, `DEPFREQ_A` e `DEPLEVEL_A` registrarão histórico, frequência e intensidade autorrelatados; `DEPMED_A` indicará o uso de medicação para depressão. `PHQ81_A` a `PHQ88_A` abrangerão os sintomas perguntados nos últimos 14 dias, e `PHQCAT_A` representará a classificação de severidade dos sintomas pela escala PHQ. Entre esses itens, `PHQ88_A` abordará alterações na velocidade da fala ou dos movimentos.

A manutenção dos itens e da categoria de severidade na base analítica permitirá explorar perfis. Como `PHQCAT_A` será derivada dos itens PHQ, a utilização simultânea de ambos poderá dar peso duplicado a essa dimensão. Para a clusterização, serão comparadas soluções com os itens e com a categoria, a escolha será documentada e as variáveis ordinais serão preparadas de acordo com o método. `LONELY_A` será mantida como variável de contexto psicossocial: ela medirá solidão, não um sintoma específico da escala PHQ.

Na modelagem supervisionada, `PHQCAT_A` e os itens que a compõem serão excluídos dos preditores quando a própria categoria ou um diagnóstico de depressão for o alvo, pois revelariam informações usadas na definição do desfecho. Histórico, frequência, intensidade e uso de medicação também serão avaliados conforme o alvo e o momento em que os preditores estarão disponíveis.

**Organização da seleção:** `GAD72_A` corresponderá a uma pergunta da escala GAD sobre dificuldade de controlar a preocupação. Por isso, será mantida somente no bloco de ansiedade, separada dos itens PHQ.

In [22]:
HISTORICO_DEPRESSAO= "HISTORICO_DEPRESSAO"
REMEDIO_DEPRESSAO= "REMEDIO_DEPRESSAO"
FREQ_DEPRESSAO= "FREQ_DEPRESSAO"
INTENSIDADE_DEPRESSAO= "INTENSIDADE_DEPRESSAO"
FREQ_PERDA_INTERESSE_2SEM= "FREQ_PERDA_INTERESSE_2SEM"
FREQ_SENTIU_DEPRIMIDO_2SEM= "FREQ_SENTIU_DEPRIMIDO_2SEM"


FREQ_DIFICULDADE_DORMIR_2SEM= "FREQ_DIFICULDADE_DORMIR_2SEM"
FREQ_CANSASO_2SEM= "FREQ_CANSASO_2SEM"
FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM= "FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM"
FREQ_AUTOAVALIACAO_NEGTIVA_2SEM= "FREQ_AUTOAVALIACAO_NEGTIVA_2SEM"
FREQ_DIFICULDADE_CONCENTRACAO_2SEM= "FREQ_DIFICULDADE_CONCENTRACAO_2SEM"


FREQ_SOLITARIO= "FREQ_SOLITARIO"


FREQUENCIA_FALHA_MEMORIA= "FREQUENCIA_FALHA_MEMORIA"
TIPO_DIFICULDADE_COGNITIVA= "TIPO_DIFICULDADE_COGNITIVA"
QTD_DIFICULDADE_COGNITIVA= "QTD_DIFICULDADE_COGNITIVA"

MODIFICACAO_VELOCIDADE_FALA = "MODIFICACAO_VELOCIDADE_FALA"

SEVERIDADE_DEPRESSAO_PHQ = "SEVERIDADE_DEPRESSAO_PHQ"


MAPA_VARIAVEIS_PAPER.update(
    {
    #Ansiedade, Depressão e seus Sintomas ########################################################

    "DEPEV_A": HISTORICO_DEPRESSAO,
    "DEPMED_A": REMEDIO_DEPRESSAO, #pode enviesar muito!
    "DEPFREQ_A": FREQ_DEPRESSAO,
    "DEPLEVEL_A": INTENSIDADE_DEPRESSAO,
    "PHQ81_A": FREQ_PERDA_INTERESSE_2SEM,
    "PHQ82_A": FREQ_SENTIU_DEPRIMIDO_2SEM,
    #"HOPELESS_A": "k6_freq_desesperanca", #infelizmente não tem 2025. PHQ82_A equivalente?
    #"WORTHLESS_A": "k6_freq_desvalorizacao", #infelizmente não tem 2025. PHQ82_A equivalente?
    "PHQ83_A": FREQ_DIFICULDADE_DORMIR_2SEM,
    #"SLPFLL_A": "dificuldade_adormecer", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPSTY_A": "dificuldade_manter_sono", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    #"SLPREST_A": "acorda_descansado", #infelizmente não tem no paper e 2025. PHQ83_A equivalente?
    "PHQ84_A": FREQ_CANSASO_2SEM,
    #"FGEFRQTRD_A": "frequencia_fadiga_3m", #infelizmente não tem no paper e 2025. PHQ84_A equivalente?
    "PHQ85_A": FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM,
    "PHQ86_A": FREQ_AUTOAVALIACAO_NEGTIVA_2SEM, #auto estima, etc...
    "PHQ87_A": FREQ_DIFICULDADE_CONCENTRACAO_2SEM,
    #"SAD_A": "k6_freq_tristeza_profunda", #infelizmente não tem 2025. PHQ82_A equivalente?
    "PHQ88_A": MODIFICACAO_VELOCIDADE_FALA, #infelizmente não tem no paper.
    "PHQCAT_A": SEVERIDADE_DEPRESSAO_PHQ,


    "LONELY_A": FREQ_SOLITARIO, #não está no paper mas deveria
    #"EFFORT_A": "k6_freq_esforco_extremo", não está no paper. Não achei equivalente
    #"PHQCAT_A": "escala_sintomas_depressão", #pode enviesar muito!


    }
)



### Seleção das variáveis de memória e dificuldade cognitiva

`COGFRQDFF_A` registrará a frequência de dificuldade para lembrar; `COGTYPEDFF_A` distinguirá dificuldade para lembrar, para se concentrar ou para ambas; `COGMEMDFF_A` medirá o grau dessa dificuldade, de nenhuma a incapacidade de realizá-la. As três variáveis serão mantidas para permitir analisar frequência, tipo e intensidade como dimensões complementares.

Como o tipo e o grau da dificuldade serão perguntas de acompanhamento, poderão não se aplicar a participantes que não relatarem dificuldade. As ausências estruturais serão tratadas conforme o fluxo do questionário e diferenciadas de respostas desconhecidas. O nome `QTD_DIFICULDADE_COGNITIVA` poderá sugerir uma contagem, embora `COGMEMDFF_A` represente grau ou severidade; essa nomenclatura será revista de forma consistente nos notebooks antes das análises.

Essas variáveis poderão complementar a clusterização dos perfis de saúde mental, embora não sejam indicadores diretos de ansiedade ou depressão. Na etapa supervisionada, sua inclusão será definida conforme o alvo e a pergunta de pesquisa. Se a previsão anteceder a medição cognitiva ou se o alvo incorporar essas respostas, elas serão consideradas indisponíveis ou como possíveis fontes de vazamento.

In [23]:

FREQUENCIA_FALHA_MEMORIA= "FREQUENCIA_FALHA_MEMORIA"
TIPO_DIFICULDADE_COGNITIVA= "TIPO_DIFICULDADE_COGNITIVA"
QTD_DIFICULDADE_COGNITIVA= "QTD_DIFICULDADE_COGNITIVA"


MAPA_VARIAVEIS_PAPER.update(
    {

    "COGFRQDFF_A": FREQUENCIA_FALHA_MEMORIA, 
    "COGTYPEDFF_A": TIPO_DIFICULDADE_COGNITIVA,
    "COGMEMDFF_A": QTD_DIFICULDADE_COGNITIVA,

    }
)


In [24]:

# 1. Dicionário Consolidado das Variáveis Selecionadas : MAPA_VARIAVEIS_PAPER (72 variáveis selecionadas)


colunas_para_ler = list(MAPA_VARIAVEIS_PAPER.keys())
print(f"Colunas a serem carregadas: {len(colunas_para_ler)}")

df_selecionado = df_bruto[colunas_para_ler]

# 4. Aplicação dos Nomes Padronizados em Português
df_selecionado = df_selecionado.rename(columns=MAPA_VARIAVEIS_PAPER)

# 5. Exportação para CSV selecionados
nome_arquivo_saida = "**/dados/dados_selecionados/adult25_selecionados.csv"
# Corrige o caminho e garante que o diretório exista antes de salvar
output_dir = Path("dados/dados_selecionados")
if not output_dir.exists():
    output_dir = Path("..") / "dados" / "dados_selecionados"

output_dir.mkdir(parents=True, exist_ok=True)

nome_arquivo_saida = output_dir / "adult25_selecionados.csv"
df_selecionado.to_csv(nome_arquivo_saida, index=False)

print(f"Arquivo salvo em: {nome_arquivo_saida}")

print(f"\nBase limpa exportada como '{nome_arquivo_saida}' com sucesso!")
print(f"Dimensões finais: {df_selecionado.shape[0]:,} participantes e {df_selecionado.shape[1]} atributos.")

Colunas a serem carregadas: 72
Arquivo salvo em: ..\dados\dados_selecionados\adult25_selecionados.csv

Base limpa exportada como '..\dados\dados_selecionados\adult25_selecionados.csv' com sucesso!
Dimensões finais: 24,215 participantes e 72 atributos.


In [25]:
MAPA_VARIAVEIS_PAPER

{'HHX': 'ID_DOMICILIO',
 'WTFA_A': 'PESO_AMOSTRAL',
 'SEX_A': 'SEXO',
 'URBRRL23': 'CLASSIFICACAO_URBANO_RURAL',
 'REGION': 'REGIAO_GEOGRAFICA',
 'MARITAL_A': 'ESTADO_CIVIL_DECLARADO',
 'MARSTAT_A': 'ESTADO_CIVIL',
 'PARSTAT_A': 'STATUS_PARENTAL',
 'SPOUSESEX_A': 'SEXO_CONJUGE',
 'SPOUSEP_A': 'SEPARACAO_LEGAL_CONJUGAL',
 'SPOUSWRK_A': 'COJUGE_TRABALHA',
 'OVER65FLG_A': 'FLG_IDOSO_FAMILIA',
 'PCNTADLT_A': 'QTD_ADULTOS_FAMILIA',
 'PCNTKIDS_A': 'QTD_CRIANÇAS_FAMILIA',
 'MAXEDUCP_A': 'ESCOLARIDADE_MAXIMA_FAMILIA',
 'HOUYRSLIV_A': 'TEMPO_MORADIA_ANOS',
 'HOUTENURE_A': 'TIPO_POSSE_IMOVEL',
 'SUPPORT_A': 'SUPORTE_SOCIAL',
 'NATUSBORN_A': 'NATURAL_ESTADOS_UNIDOS',
 'PHSTAT_A': 'AUTOAVALIACAO_SAUDE_GERAL',
 'BMICAT_A': 'CATEGORIA_IMC',
 'CHLEV_A': 'HISTORICO_COLESTEROL',
 'DIBEV_A': 'HISTORICO_DIABETES',
 'HYPEV_A': 'HISTORICO_HIPERTENSAO',
 'ARTHEV_A': 'HISTORICO_ARTRITE',
 'TBIHLSBMC_A': 'CONCUSSAO_SINTOMAS_POS_TRAUMA',
 'DIFF_A': 'DIFICULDADE_LOCOMOCAO',
 'SOCWRKLIM_A': 'LIMITACAO_TRABALHO_P